In [1]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/katabatic1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/katabatic1


In [4]:
!pip install loguru


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.5 MB/s eta 0:00:00


In [3]:
!pip install pytorch-lightning==2.1.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 777.7/777.7 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 66.8 MB/s eta 0:00:00


In [1]:
import pytorch_lightning as pl
print(pl.__version__)


2.1.3


In [13]:
import time
from pathlib import Path
import pandas as pd

from katabatic.utils.split_dataset import split_dataset
from katabatic.models.decaf.adapter import DECAFModel
from katabatic.evaluate.tstr.evaluation import TSTREvaluation


start = time.time()

ROOT = Path.cwd().resolve()

RAW_CSV = ROOT / "raw_data" / "car.csv"
SAMPLE_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR = ROOT / "synthetic" / "car" / "decaf"
RESULTS_DIR = ROOT / "Results" / "car"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


print("Splitting CAR dataset")

split_dataset(
    input_csv=RAW_CSV,
    output_dir=SAMPLE_DIR,
    label_col="class",
)


print("Training DECAF (CAR, no tuning)")

model = DECAFModel()
model.train(
    data_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
)


print("Preparing DECAF synthetic data for TSTR")

# Load synthetic data generated by DECAF
synth_df = pd.read_csv(SYNTH_DIR / "synthetic.csv")

# Encode categorical synthetic features
for col in synth_df.columns:
    if not pd.api.types.is_numeric_dtype(synth_df[col]):
        synth_df[col] = synth_df[col].astype("category").cat.codes

synth_df = synth_df.astype("float32")

# Align synthetic features to real feature space
real_X_cols = pd.read_csv(SAMPLE_DIR / "x_train.csv").columns
X_synth = synth_df[real_X_cols]

# DECAF is unsupervised: derive y_synth from real y_train
y_train_real = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_synth = y_train_real.iloc[: len(X_synth), 0]

if not pd.api.types.is_numeric_dtype(y_synth):
    y_synth = y_synth.astype("category").cat.codes

y_synth = y_synth.astype("float32")

# Save synthetic files using Katabatic-expected names
X_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)


print("Encoding REAL data for TSTR")

X_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
X_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_train = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")

# Encode categorical real features
for col in X_train.columns:
    if not pd.api.types.is_numeric_dtype(X_train[col]):
        X_train[col] = X_train[col].astype("category").cat.codes
        X_test[col] = X_test[col].astype("category").cat.codes

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

# Encode labels
if not pd.api.types.is_numeric_dtype(y_train.iloc[:, 0]):
    y_train.iloc[:, 0] = y_train.iloc[:, 0].astype("category").cat.codes
if not pd.api.types.is_numeric_dtype(y_test.iloc[:, 0]):
    y_test.iloc[:, 0] = y_test.iloc[:, 0].astype("category").cat.codes

y_train = y_train.astype("float32")
y_test = y_test.astype("float32")

# Save encoded real data back
X_train.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
X_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)
y_train.to_csv(SAMPLE_DIR / "y_train.csv", index=False)
y_test.to_csv(SAMPLE_DIR / "y_test.csv", index=False)


print("Running TSTR Evaluation")

tstr = TSTREvaluation(
    real_train_dir=str(SAMPLE_DIR),
    real_test_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
    dataset_name="car",
    model_name="decaf",
)

results = tstr.evaluate()

tstr.save_results_to_csv(
    results=results,
    out_dir=str(RESULTS_DIR),
    filename="decaf_tstr.csv",
)


print("TSTR evaluation completed")
print(f"Total runtime: {(time.time() - start) / 60:.2f} minutes")


INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-01-25 16:31:57.894 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - ***** DATA *****
2026-01-25 16:31:57.895 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - n_samples = 1382
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type             | Params
---------------------------------------------------
0 | generator     | Generator_causal | 94.5 K
1 | discriminator | Discriminator    | 42.0 K
---------------------------------------------------
136 K     Trainable params
0         Non-trai

Splitting CAR dataset
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
Training DECAF (CAR, no tuning)


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.


[DECAF] Synthetic data generated
Preparing DECAF synthetic data for TSTR
Encoding REAL data for TSTR
Running TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/car/decaf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.4913
F1 Score: 0.5303

MLP:
Accuracy: 0.5289
F1 Score: 0.5386

RF:
Accuracy: 0.5954
F1 Score: 0.5963

XGBoost:
Accuracy: 0.6474
F1 Score: 0.5811


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:32:31] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


TypeError: TSTREvaluation.save_results_to_csv() got an unexpected keyword argument 'out_dir'

In [18]:
import time
from pathlib import Path
import pandas as pd

from katabatic.utils.split_dataset import split_dataset
from katabatic.models.decaf.adapter import DECAFModel
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

start = time.time()

ROOT = Path.cwd().resolve()

RAW_CSV = ROOT / "raw_data" / "magic.csv"
SAMPLE_DIR = ROOT / "sample_data" / "magic"
SYNTH_DIR = ROOT / "synthetic" / "magic" / "decaf"
RESULTS_DIR = ROOT / "Results" / "magic"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Splitting MAGIC dataset")

split_dataset(
    input_csv=RAW_CSV,
    output_dir=SAMPLE_DIR,
    label_col="class",
)

print("Training DECAF (MAGIC, no tuning)")

model = DECAFModel()
model.train(
    data_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
)

print("Preparing DECAF synthetic data for TSTR")

# Load synthetic data generated by DECAF
synth_df = pd.read_csv(SYNTH_DIR / "synthetic.csv")

# Drop label column — CRITICAL FIX
synth_df = synth_df.drop(columns=["class"])

# Ensure numeric
synth_df = synth_df.astype("float32")

# Save x_synth
synth_df.to_csv(SYNTH_DIR / "x_synth.csv", index=False)

# Build y_synth from real y_train (aligned, no tuning)
y_train = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_train.iloc[: len(synth_df), 0].to_csv(
    SYNTH_DIR / "y_synth.csv", index=False
)

print("Running TSTR Evaluation")

tstr = TSTREvaluation(
    real_train_dir=str(SAMPLE_DIR),
    real_test_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
    dataset_name="magic",
    model_name="decaf",
)

results = tstr.evaluate()

print("TSTR evaluation completed")
print(f"Total runtime: {(time.time() - start) / 60:.2f} minutes")


INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-01-25 16:57:40.695 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - ***** DATA *****
2026-01-25 16:57:40.696 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - n_samples = 15216


Splitting MAGIC dataset
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
Training DECAF (MAGIC, no tuning)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type             | Params
---------------------------------------------------
0 | generator     | Generator_causal | 111 K 
1 | discriminator | Discriminator    | 42.8 K
---------------------------------------------------
154 K     Trainable params
0         Non-trainable params
154 K     Total params
0.617     Total estimated model params size (MB)
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottlene

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.


[DECAF] Synthetic data generated
Preparing DECAF synthetic data for TSTR
Running TSTR Evaluation

Results saved to: Results/magic/decaf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6725
F1 Score: 0.6467
AUC: 0.6119

MLP:
Accuracy: 0.5523
F1 Score: 0.5056
AUC: 0.3344

RF:
Accuracy: 0.4393
F1 Score: 0.4519
AUC: 0.4287

XGBoost:
Accuracy: 0.5991
F1 Score: 0.5654
AUC: 0.4467
TSTR evaluation completed
Total runtime: 7.28 minutes


In [21]:
import time
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from katabatic.utils.split_dataset import split_dataset
from katabatic.models.decaf.adapter import DECAFModel
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

start = time.time()

ROOT = Path.cwd().resolve()

RAW_CSV = ROOT / "raw_data" / "nursery.csv"
SAMPLE_DIR = ROOT / "sample_data" / "nursery"
SYNTH_DIR = ROOT / "synthetic" / "nursery" / "decaf"
RESULTS_DIR = ROOT / "Results" / "nursery"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Splitting NURSERY dataset")

split_dataset(
    input_csv=RAW_CSV,
    output_dir=SAMPLE_DIR,
    label_col="class",
)

print("Encoding labels ONCE (global mapping)")

le = LabelEncoder()

y_train = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_test  = pd.read_csv(SAMPLE_DIR / "y_test.csv")

y_train.iloc[:, 0] = le.fit_transform(y_train.iloc[:, 0])
y_test.iloc[:, 0]  = le.transform(y_test.iloc[:, 0])

y_train.to_csv(SAMPLE_DIR / "y_train.csv", index=False)
y_test.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Training DECAF (NURSERY, no tuning)")

model = DECAFModel()
model.train(
    data_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
)

print("Preparing DECAF synthetic data for TSTR")

synth_df = pd.read_csv(SYNTH_DIR / "synthetic.csv")

for col in synth_df.columns:
    if not pd.api.types.is_numeric_dtype(synth_df[col]):
        synth_df[col] = synth_df[col].astype("category").cat.codes

synth_df = synth_df.astype("float32")

X_synth = synth_df
y_train_encoded = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_synth = y_train_encoded.iloc[:len(X_synth), 0]

X_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Encoding REAL feature data for TSTR")

X_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
X_test  = pd.read_csv(SAMPLE_DIR / "x_test.csv")

for col in X_train.columns:
    if not pd.api.types.is_numeric_dtype(X_train[col]):
        X_train[col] = X_train[col].astype("category").cat.codes
        X_test[col]  = X_test[col].astype("category").cat.codes

X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")

X_train.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
X_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Forcing feature alignment (critical for NURSERY)")

X_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

X_train.columns = X_train.columns.astype(str)
X_test.columns  = X_test.columns.astype(str)
X_synth.columns = X_synth.columns.astype(str)

X_synth = X_synth.reindex(columns=X_train.columns).fillna(0.0)
X_test  = X_test.reindex(columns=X_train.columns).fillna(0.0)

assert list(X_train.columns) == list(X_test.columns) == list(X_synth.columns)

X_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)

print("Running TSTR Evaluation")

tstr = TSTREvaluation(
    real_train_dir=str(SAMPLE_DIR),
    real_test_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
    dataset_name="nursery",
    model_name="decaf",
)

results = tstr.evaluate()

print("Results saved to:", RESULTS_DIR / "decaf_tstr.csv")
print("TSTR evaluation completed")
print(f"Total runtime: {(time.time() - start) / 60:.2f} minutes")


Splitting NURSERY dataset
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
Encoding labels ONCE (global mapping)


INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-01-25 17:22:33.766 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - ***** DATA *****
2026-01-25 17:22:33.767 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - n_samples = 10368
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type             | Params
---------------------------------------------------
0 | generator     | Generator_causal | 102 K 
1 | discriminator | Discriminator    | 42.4 K
---------------------------------------------------
144 K     Trainable params
0         Non-tra

Training DECAF (NURSERY, no tuning)


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.


[DECAF] Synthetic data generated
Preparing DECAF synthetic data for TSTR
Encoding REAL feature data for TSTR
Forcing feature alignment (critical for NURSERY)
Running TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:26:48] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/nursery/decaf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.2558
F1 Score: 0.2086

MLP:
Accuracy: 0.1848
F1 Score: 0.1797

RF:
Accuracy: 0.1250
F1 Score: 0.1102

XGBoost:
Accuracy: 0.2604
F1 Score: 0.2207
Results saved to: /content/drive/MyDrive/katabatic1/Results/nursery/decaf_tstr.csv
TSTR evaluation completed
Total runtime: 4.25 minutes


In [22]:
import time
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from katabatic.utils.split_dataset import split_dataset
from katabatic.models.decaf.adapter import DECAFModel
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

start = time.time()

ROOT = Path.cwd().resolve()

RAW_CSV = ROOT / "raw_data" / "shuttle.csv"
SAMPLE_DIR = ROOT / "sample_data" / "shuttle"
SYNTH_DIR = ROOT / "synthetic" / "shuttle" / "decaf"
RESULTS_DIR = ROOT / "Results" / "shuttle"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Splitting SHUTTLE dataset")

split_dataset(
    input_csv=RAW_CSV,
    output_dir=SAMPLE_DIR,
    label_col="class",
)

print("Encoding labels ONCE (global mapping)")

le = LabelEncoder()

y_train = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_test  = pd.read_csv(SAMPLE_DIR / "y_test.csv")

y_train.iloc[:, 0] = le.fit_transform(y_train.iloc[:, 0])
y_test.iloc[:, 0]  = le.transform(y_test.iloc[:, 0])

y_train.to_csv(SAMPLE_DIR / "y_train.csv", index=False)
y_test.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Training DECAF (SHUTTLE, no tuning)")

model = DECAFModel()
model.train(
    data_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
)

print("Preparing DECAF synthetic data for TSTR")

synth_df = pd.read_csv(SYNTH_DIR / "synthetic.csv")

for col in synth_df.columns:
    if not pd.api.types.is_numeric_dtype(synth_df[col]):
        synth_df[col] = synth_df[col].astype("category").cat.codes

synth_df = synth_df.astype("float32")

X_synth = synth_df
y_train_encoded = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_synth = y_train_encoded.iloc[:len(X_synth), 0]

X_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Encoding REAL feature data for TSTR")

X_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
X_test  = pd.read_csv(SAMPLE_DIR / "x_test.csv")

for col in X_train.columns:
    if not pd.api.types.is_numeric_dtype(X_train[col]):
        X_train[col] = X_train[col].astype("category").cat.codes
        X_test[col]  = X_test[col].astype("category").cat.codes

X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")

X_train.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
X_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Forcing feature alignment (SHUTTLE-safe)")

X_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

X_train.columns = X_train.columns.astype(str)
X_test.columns  = X_test.columns.astype(str)
X_synth.columns = X_synth.columns.astype(str)

X_synth = X_synth.reindex(columns=X_train.columns).fillna(0.0)
X_test  = X_test.reindex(columns=X_train.columns).fillna(0.0)

assert list(X_train.columns) == list(X_test.columns) == list(X_synth.columns)

X_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)

print("Running TSTR Evaluation")

tstr = TSTREvaluation(
    real_train_dir=str(SAMPLE_DIR),
    real_test_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
    dataset_name="shuttle",
    model_name="decaf",
)

results = tstr.evaluate()

print("Results saved to:", RESULTS_DIR / "decaf_tstr.csv")
print("TSTR evaluation completed")
print(f"Total runtime: {(time.time() - start) / 60:.2f} minutes")


Splitting SHUTTLE dataset
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64


INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-01-25 17:35:10.087 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - ***** DATA *****
2026-01-25 17:35:10.087 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - n_samples = 46400
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type             | Params
---------------------------------------------------
0 | generator     | Generator_causal | 106 K 
1 | discriminator | Discriminator    | 42.6 K
---------------------------------------------------
149 K     Trainable params
0         Non-tra

Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
Encoding labels ONCE (global mapping)
Training DECAF (SHUTTLE, no tuning)


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.


[DECAF] Synthetic data generated
Preparing DECAF synthetic data for TSTR
Encoding REAL feature data for TSTR
Forcing feature alignment (SHUTTLE-safe)
Running TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:55:58] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/shuttle/decaf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.2626
F1 Score: 0.3810

MLP:
Accuracy: 0.5760
F1 Score: 0.6335

RF:
Accuracy: 0.2336
F1 Score: 0.2408

XGBoost:
Accuracy: 0.5082
F1 Score: 0.5660
Results saved to: /content/drive/MyDrive/katabatic1/Results/shuttle/decaf_tstr.csv
TSTR evaluation completed
Total runtime: 20.96 minutes


In [23]:
import time
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from katabatic.utils.split_dataset import split_dataset
from katabatic.models.decaf.adapter import DECAFModel
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

start = time.time()

ROOT = Path.cwd().resolve()

RAW_CSV = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR = ROOT / "synthetic" / "adult" / "decaf"
RESULTS_DIR = ROOT / "Results" / "adult"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Splitting ADULT dataset")

split_dataset(
    input_csv=RAW_CSV,
    output_dir=SAMPLE_DIR,
    label_col="class",
)

print("Encoding labels ONCE (global mapping)")

le = LabelEncoder()

y_train = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_test  = pd.read_csv(SAMPLE_DIR / "y_test.csv")

y_train.iloc[:, 0] = le.fit_transform(y_train.iloc[:, 0])
y_test.iloc[:, 0]  = le.transform(y_test.iloc[:, 0])

y_train.to_csv(SAMPLE_DIR / "y_train.csv", index=False)
y_test.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Training DECAF (ADULT, no tuning)")

model = DECAFModel()
model.train(
    data_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
)

print("Preparing DECAF synthetic data for TSTR")

synth_df = pd.read_csv(SYNTH_DIR / "synthetic.csv")

for col in synth_df.columns:
    if not pd.api.types.is_numeric_dtype(synth_df[col]):
        synth_df[col] = synth_df[col].astype("category").cat.codes

synth_df = synth_df.astype("float32")

X_synth = synth_df
y_train_encoded = pd.read_csv(SAMPLE_DIR / "y_train.csv")
y_synth = y_train_encoded.iloc[:len(X_synth), 0]

X_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Encoding REAL feature data for TSTR")

X_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
X_test  = pd.read_csv(SAMPLE_DIR / "x_test.csv")

for col in X_train.columns:
    if not pd.api.types.is_numeric_dtype(X_train[col]):
        X_train[col] = X_train[col].astype("category").cat.codes
        X_test[col]  = X_test[col].astype("category").cat.codes

X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")

X_train.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
X_test.to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Forcing feature alignment (ADULT-safe)")

X_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

X_train.columns = X_train.columns.astype(str)
X_test.columns  = X_test.columns.astype(str)
X_synth.columns = X_synth.columns.astype(str)

X_synth = X_synth.reindex(columns=X_train.columns).fillna(0.0)
X_test  = X_test.reindex(columns=X_train.columns).fillna(0.0)

assert list(X_train.columns) == list(X_test.columns) == list(X_synth.columns)

X_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)

print("Running TSTR Evaluation")

tstr = TSTREvaluation(
    real_train_dir=str(SAMPLE_DIR),
    real_test_dir=str(SAMPLE_DIR),
    synthetic_dir=str(SYNTH_DIR),
    dataset_name="adult",
    model_name="decaf",
)

results = tstr.evaluate()

print("Results saved to:", RESULTS_DIR / "decaf_tstr.csv")
print("TSTR evaluation completed")
print(f"Total runtime: {(time.time() - start) / 60:.2f} minutes")


Splitting ADULT dataset
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64


INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-01-25 17:59:27.486 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - ***** DATA *****
2026-01-25 17:59:27.487 | INFO     | katabatic.models.decaf.logger:log_and_print:66 - n_samples = 26048
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type             | Params
---------------------------------------------------
0 | generator     | Generator_causal | 134 K 
1 | discriminator | Discriminator    | 43.6 K
---------------------------------------------------
178 K     Trainable params
0         Non-tra

Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Encoding labels ONCE (global mapping)
Training DECAF (ADULT, no tuning)


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.


[DECAF] Synthetic data generated
Preparing DECAF synthetic data for TSTR
Encoding REAL feature data for TSTR
Forcing feature alignment (ADULT-safe)
Running TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/decaf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.2193
F1 Score: 0.0917
AUC: 0.4427

MLP:
Accuracy: 0.2291
F1 Score: 0.0916
AUC: 0.4742

RF:
Accuracy: 0.6622
F1 Score: 0.6206
AUC: 0.4420

XGBoost:
Accuracy: 0.7593
F1 Score: 0.6553
AUC: 0.4288
Results saved to: /content/drive/MyDrive/katabatic1/Results/adult/decaf_tstr.csv
TSTR evaluation completed
Total runtime: 15.94 minutes
